# Part 3: Scalable Processing — Polars Query Optimization, Pandas Benchmark & PySpark Parity
Examines Polars lazy query optimization (predicate pushdown, projection pushdown, parallel window execution), benchmarks execution time and peak RSS against Pandas (naive vs column-pushdown), and verifies exact parity with PySpark.

In [1]:
from pathlib import Path
import polars as pl

print('=== Naive vs Optimized Polars Query Plan ===')
print(Path('../outputs/tables/polars_naive_vs_optimized_plan.txt').read_text(encoding='utf-8'))

=== Naive vs Optimized Polars Query Plan ===
=== NAIVE LOGICAL PLAN (UNPROJECTED, POST-CORE FILTER ON COMPUTED `year`) ===
FILTER col("year") >= 2008
FROM
   WITH_COLUMNS:
   [when(col("qidx").shift([dyn int: 4]).over([col("CERT")]) == (col("qidx") - 4)).then((col("ASSET") / col("ASSET").shift([dyn int: 4]).over([col("CERT")])) - 1.0).otherwise(null.cast(Float64)).alias("asset_growth_4q"), when(col("qidx").shift([dyn int: 4]).over([col("CERT")]) == (col("qidx") - 4)).then(col("texas_ratio") - col("texas_ratio").shift([dyn int: 4]).over([col("CERT")])).otherwise(null.cast(Float64)).alias("d4_texas_ratio")] 
     WITH_COLUMNS:
     [when(col("quarter") == 1).then(col("NETINC")).otherwise(when(col("qidx").shift([dyn int: 1]).over([col("CERT"), col("year")]) == (col("qidx") - 1)).then(col("NETINC") - col("NETINC").shift([dyn int: 1]).over([col("CERT"), col("year")])).otherwise(null.cast(Float64))).alias("NETINC_Q"), (col("EQ") / col("ASSET")).alias("eq_asset_ratio"), when((col("EQ").fill_n

In [2]:
bench = pl.read_csv('../outputs/tables/part3_benchmark_results.csv')
print('=== Part 3 Multi-Scale (1x-8x) Benchmark ===')
print(bench)
print('\n=== Full 34-Col Bronze -> 67-Col Gold Benchmark (collect vs sink_parquet) ===')
print(Path('../outputs/tables/part3_full_schema_benchmark.json').read_text(encoding='utf-8'))
print('\n=== PySpark Multi-Scale (1x & 4x) Python+JVM RSS ===')
print(pl.read_csv('../outputs/tables/spark_scaling_results.csv'))

=== Part 3 Multi-Scale (1x-8x) Benchmark ===
shape: (15, 7)
┌───────┬──────────────┬──────────┬───────────────────┬────────┬─────────────┬─────────────────────┐
│ scale ┆ scale_factor ┆ rows     ┆ engine            ┆ time_s ┆ peak_rss_mb ┆ max_rel_diff_vs_pol │
│ ---   ┆ ---          ┆ ---      ┆ ---               ┆ ---    ┆ ---         ┆ ars_lazy            │
│ str   ┆ i64          ┆ i64      ┆ str               ┆ f64    ┆ f64         ┆ ---                 │
│       ┆              ┆          ┆                   ┆        ┆             ┆ f64                 │
╞═══════╪══════════════╪══════════╪═══════════════════╪════════╪═════════════╪═════════════════════╡
│ 1x    ┆ 1            ┆ 1682616  ┆ Polars lazy       ┆ 1.326  ┆ 777.0       ┆ 0.0                 │
│ 1x    ┆ 1            ┆ 1682616  ┆ Polars streaming  ┆ 0.619  ┆ 1268.0      ┆ 2.6000e-16          │
│ 1x    ┆ 1            ┆ 1682616  ┆ Polars eager      ┆ 1.281  ┆ 711.0       ┆ 0.0                 │
│ 1x    ┆ 1            ┆ 168261

In [3]:
import json
parity = json.loads(Path('../outputs/tables/spark_polars_parity.json').read_text(encoding='utf-8'))
print('=== PySpark vs Polars Parity Check ===')
print(json.dumps(parity, indent=2))
print('\n=== PySpark Catalyst Execution Plan (Excerpt) ===')
print(Path('../outputs/tables/spark_execution_plan.txt').read_text(encoding='utf-8')[:1500])

=== PySpark vs Polars Parity Check ===
{
  "yearly_fingerprint": {
    "years_compared": 43,
    "row_and_null_counts_identical": true,
    "max_relative_diff_of_sums": 7.424666459363888e-15
  },
  "row_level_sample": {
    "banks_sampled": 483,
    "rows_compared": 33042,
    "rows_only_in_one_engine": 0,
    "max_abs_diff_any_column": 0.0
  },
  "timing_seconds": {
    "polars_lazy_collect_plus_aggregate": 0.67,
    "polars_threads": 4,
    "spark_session_startup": 4.22,
    "spark_job_cold": 5.02,
    "spark_job_warm": 2.07,
    "note": "single in-process runs on the full panel; Polars = 4 threads, Spark = local[2], 1.2 GB driver"
  }
}

=== PySpark Catalyst Execution Plan (Excerpt) ===
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- Window (10)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Window (6)
                  +- Sort (5)
                     +- Exchange (4)
                        +- Project (3)
                   